# Análise de Outliers e Contratos Potencialmente Suspeitos



#### Imports e leitura

In [2]:
import pandas as pd
import numpy as np

file_path = "contratos_limpo.csv"
df = pd.read_csv(file_path, low_memory=False)

print("Nº total de registos:", len(df))
df.columns

Nº total de registos: 1803493


Index(['idcontrato', 'tipoContrato', 'idprocedimento', 'tipoprocedimento',
       'objectoContrato', 'descContrato', 'adjudicante', 'adjudicatarios',
       'dataPublicacao', 'dataCelebracaoContrato', 'precoContratual', 'CPV',
       'prazoExecucao', 'LocalExecucao', 'fundamentacao',
       'ProcedimentoCentralizado', 'DescrAcordoQuadro',
       'dataDecisaoAdjudicacao', 'regime', 'justifNReducEscrContrato',
       'CritMateriais', 'concorrentes', 'linkPecasProc', 'Observacoes',
       'ContratEcologico', 'fundamentAjusteDireto', 'Ano', 'ano_contrato',
       'precoContratual_2025', 'setor_cpv', 'descricao_cpv',
       'racio_preco_prazo', 'janela_eleicoes', 'AR', 'AL'],
      dtype='str')

##  Preparação dos dados

In [3]:
 #DESCRIÇÃO POR SETOR

descricoes_setor = (
    df.dropna(subset=['setor_cpv'])
      .groupby('setor_cpv')
      .agg(
          descricao_setor=(
              'descricao_cpv',
              lambda x: x.dropna().mode().iat[0] if not x.dropna().mode().empty else np.nan
          )
      )
      .reset_index()
)


# DETEÇÃO DE OUTLIERS (IQR)

def detectar_outliers_racio_por_setor(df_in, col_setor='setor_cpv', col_racio='racio_preco_prazo', fator_iqr=1.5):
    df_aux = df_in.copy()
    df_aux = df_aux.dropna(subset=[col_setor, col_racio])

    q1 = df_aux.groupby(col_setor)[col_racio].transform(lambda x: x.quantile(0.25))
    q3 = df_aux.groupby(col_setor)[col_racio].transform(lambda x: x.quantile(0.75))
    iqr = q3 - q1

    li = q1 - fator_iqr * iqr
    ls = q3 + fator_iqr * iqr

    df_aux['outlier_racio'] = (df_aux[col_racio] < li) | (df_aux[col_racio] > ls)
    df_aux['limite_inferior_setor'] = li
    df_aux['limite_superior_setor'] = ls

    return df_aux

# RESUMO (COM FILTRO > 100)

def resumo_outliers_por_setor(df_outliers, nome_setor='setor_cpv'):
    resumo = (
        df_outliers.groupby(nome_setor)
        .agg(
            n_registos=(nome_setor, 'size'),
            n_outliers=('outlier_racio', 'sum')
        )
        .reset_index()
    )

    resumo['perc_outliers_setor'] = (
        resumo['n_outliers'] / resumo['n_registos'] * 100
    ).round(2)

    #Filtro dos 100
    resumo = resumo[resumo['n_registos'] > 100]

    # merge com descrições
    resumo = resumo.merge(
        descricoes_setor,
        on='setor_cpv',
        how='left'
    )

    return resumo.sort_values('perc_outliers_setor', ascending=False)

## Análise dos contratos com prazoExecucao > 1

In [4]:
# subconjunto com prazo > 1
df_maior1 = df[df['prazoExecucao'] > 1].copy()

df_outliers_maior1 = detectar_outliers_racio_por_setor(df_maior1)

outliers_maior1 = df_outliers_maior1[df_outliers_maior1['outlier_racio']].copy()

resumo_outliers_racio = resumo_outliers_por_setor(df_outliers_maior1)

In [5]:
print('Nº de registos com prazo > 1:', len(df_maior1))
print('Nº de outliers com prazo > 1:', len(outliers_maior1))

Nº de registos com prazo > 1: 1683169
Nº de outliers com prazo > 1: 203726


In [6]:
resumo_outliers_racio.head(5)

,setor_cpv,n_registos,n_outliers,perc_outliers_setor,descricao_setor
30,65.0,6427,993,15.45,Serviços públicos
44,98.0,29559,4466,15.11,Serviços diversos
43,92.0,51647,7518,14.56,"Serviços recreativos, culturais e desportivos"
39,79.0,109264,15519,14.20,"Serviços a empresas: direito, comercialização,..."
12,33.0,471957,66321,14.05,Produtos farmacêuticos


In [7]:
resumo_outliers_racio.tail(5)

,setor_cpv,n_registos,n_outliers,perc_outliers_setor,descricao_setor
19,42.0,15341,1381,9.00,Máquinas industriais
15,37.0,5735,511,8.91,"Instrumentos musicais, artigos de desporto, jo..."
33,71.0,75579,6683,8.84,"Serviços de arquitectura, construção, engenhar..."
4,16.0,2493,219,8.78,Maquinaria agrícola
22,45.0,138472,10573,7.64,Construção


## Análise dos contratos com prazoExecucao = 1

In [8]:
# subconjunto com  prazo = 1
df_prazo1 = df[df['prazoExecucao'] == 1].copy()

df_outliers_prazo1 = detectar_outliers_racio_por_setor(df_prazo1)

outliers_prazo1 = df_outliers_prazo1[df_outliers_prazo1['outlier_racio']].copy()

resumo_outliers_prazo1 = resumo_outliers_por_setor(df_outliers_prazo1)

In [9]:
len(outliers_prazo1)
print('Nº de registos com prazo = 1:', len(df_prazo1))
print('Nº de outliers com prazo = 1:', len(outliers_prazo1))

Nº de registos com prazo = 1: 75731
Nº de outliers com prazo = 1: 7727


In [10]:
resumo_outliers_prazo1.head(5)

,setor_cpv,n_registos,n_outliers,perc_outliers_setor,descricao_setor
25,71.0,701,166,23.68,"Serviços de arquitectura, construção, engenhar..."
2,15.0,1529,321,20.99,"Produtos alimentares, bebidas, tabaco e produt..."
29,80.0,455,94,20.66,Serviços de ensino e formação
17,44.0,1229,235,19.12,Estruturas e materiais de construção; produtos...
24,64.0,361,61,16.90,Serviços de telecomunicações


In [11]:
resumo_outliers_prazo1.tail(5)

,setor_cpv,n_registos,n_outliers,perc_outliers_setor,descricao_setor
15,39.0,1070,79,7.38,"Mobiliário (incl. de escritório), acessórios, ..."
33,98.0,1241,91,7.33,Serviços diversos
13,37.0,167,10,5.99,"Instrumentos musicais, artigos de desporto, jo..."
3,18.0,351,17,4.84,Equipamento (vestuário) de protecção
21,55.0,2727,86,3.15,Serviços de fornecimento de refeições (catering)


### Análise de todas o conjunto de outliers

In [13]:
#junção dos dois dfs
df_outliers = pd.concat([outliers_maior1, outliers_prazo1], axis=0)

In [22]:
len(df_outliers)

211453

#### criação de novas variáveis

In [15]:
# número de concorrentes
df_outliers["num_concorrentes"] = (
    df_outliers["concorrentes"].fillna("").astype(str)
    .str.split("\n")
    .apply(lambda lst: sum(1 for x in lst if x.strip() != ""))
)

# is_ajuste_direto
df_outliers["is_ajuste_direto"] = (
    df_outliers["tipoprocedimento"].astype(str)
    .str.contains("Ajuste Direto", case=False, na=False)
    .astype(int)
)

# dias entre decisão de publicação e adjudicação do contrato
df_outliers["dataDecisaoAdjudicacao"] = pd.to_datetime(
    df_outliers["dataDecisaoAdjudicacao"], errors="coerce"
)
df_outliers["dataPublicacao"] = pd.to_datetime(
    df_outliers["dataPublicacao"], errors="coerce"
)
df_outliers["dias_publicacao_decisao"] = (
    ( df_outliers["dataPublicacao"]- df_outliers["dataDecisaoAdjudicacao"])
    .dt.days
    .clip(lower=0)
)

In [24]:
summary_outliers = pd.DataFrame({
    'Nº de Observações': [len(df_outliers)],

    'Ano': [df_outliers['Ano'].value_counts().idxmax()],
    'Ano %': [(df_outliers['Ano'].value_counts(normalize=True).max() * 100).round(1)],

    'Setor': [df_outliers['setor_cpv'].value_counts().idxmax()],
    'Setor %': [(df_outliers['setor_cpv'].value_counts(normalize=True).max() * 100).round(1)],

    'Eleição %': [(
        df_outliers['janela_eleicoes']
        .isin(['3 meses antes', '3 meses depois'])
        .mean() * 100
    ).round(1)],

    'Ajuste Direto %': [(df_outliers['is_ajuste_direto'].mean() * 100).round(1)],

    'Nº de Concorrentes': [df_outliers['num_concorrentes'].mean().round(0)],

    'D-P': [df_outliers['dias_publicacao_decisao'].mean().round(0)],

    'Preco/Prazo': [df_outliers['racio_preco_prazo'].mean().round(0)],

    'Prazo': [df_outliers['prazoExecucao'].mean().round(0)],
})

summary_outliers.index = ['Outliers']

summary_outliers

,Nº de Observações,Ano,Ano %,Setor,Setor %,Eleição %,Ajuste Direto %,Nº de Concorrentes,D-P,Preco/Prazo,Prazo
Outliers,211453,2025,12.3,33.0,33.1,36.1,55.0,1.0,73.0,4825.0,99.0


In [23]:
#para ser utlizado no ficheiro de mans
df_outliers.to_pickle("df_outliers.pkl")